# freeCAM PI-atm

This notebook follows the same object-oriented UI as FreeCESM. Machine paths, run-directory preparation, PBS submission, `mpiexec`, and the persistent socket are implementation details hidden by `freecam.Driver`.

In [ ]:
%load_ext autoreload
%autoreload 2

import freecam as fc
import numpy as np

## 1. Create one model

Constructing the driver is cheap and does not submit a job. The first live-state operation lazily prepares a private run directory, submits one cpudev job when needed, and starts the persistent 512-rank CAM session.

In [ ]:
driver = fc.Driver(case='PI-atm', nsteps=2)
print(driver.case)
print('50-step validation:', driver.validation.get('bfb', 'see validation record'))

In [ ]:
plot_variables = (
    'T', 'u', 'v', 'q',
    'phys_state.omega', 'phys_state.pmid',
)
fig, axes = driver.cam.state.plot(
    rank=0, variables=plot_variables, figsize=(11, 6.5), label='initial'
)
print(driver.cam.state.summary(rank=0))
print('persistent job:', driver.cam.status.get('job_id', 'managed by Driver'))
print('run directory:', driver.run_dir)

## 2. Inspect and run the Python-owned workflow

The workflow is the complete per-step running order, written as one explicit Python list just like FreeCESM. Assigning the list installs that exact order atomically on every MPI rank. The default is already expanded to validated leaf processes: Python owns every ordering and alarm decision, while each row states whether its primitive is Python, a Fortran numerical kernel, or a low-level state/PIO/clock service. Internal CAM phase names are not part of this user interface; edit the order below and assign the new list.

In [ ]:
# Resolve the live process handles once, then define the complete order directly.
process = {item.operation: item for item in driver.cam.workflow}

pi_cam_workflow = [
    process['boundary_import'],
    process['prepare'],
    process['chem_emissions'],
    process['leaf_tracers_timestep_tend'],
    process['leaf_aoa_tracers_timestep_tend'],
    process['leaf_chem_timestep_tend'],
    process['vertical_diffusion_tend'],
    process['rayleigh_friction_tend'],
    process['leaf_aero_model_drydep'],
    process['leaf_carma_timestep_tend'],
    process['charge_fix'],
    process['gw_tend'],
    process['qbo_relax'],
    process['iondrag_calc'],
    process['physics_dme_adjust'],
    process['leaf_carma_accumulate_stats'],
    process['leaf_pbuf_deallocate'],
    process['leaf_pbuf_update_tim_idx'],
    process['leaf_diag_deallocate'],
    process['stepon_run2'],
    process['stepon_run3'],
    process['wshist'],
    process['restart'],
    process['leaf_cam_run4_wrapup'],
    process['leaf_cam_run4_step_cost'],
    process['leaf_cam_run4_flush'],
    process['advance_timestep'],
    process['stepon_run1'],
    process['prepare_cam_run1'],
    process['bc_init'],
    process['check_energy_fix'],
    process['dadadj'],
    process['convect_deep_tend'],
    process['convect_shallow_tend'],
    process['sslt_rebin_adv'],
    process['macro_microphysics'],
    process['leaf_modal_aero_prepare'],
    process['leaf_aero_model_wetdep'],
    process['leaf_carma_wetdep_tend'],
    process['leaf_convect_deep_tend_2'],
    process['leaf_diag_phys_writeout'],
    process['leaf_cloud_diagnostics_calc'],
    process['radiation_tend'],
    process['leaf_tropopause_output'],
    process['leaf_cam_export'],
    process['leaf_diag_export'],
    process['boundary_export'],
]

driver.cam.workflow = pi_cam_workflow
driver.cam.workflow

In [ ]:
trace = driver.execute(verbose=False)
print(f'executed {len(trace)} actions across {driver.nsteps} complete steps')
print('first process:', trace[0]['name'])
print('last process: ', trace[-1]['name'])

for axis, variable in zip(axes.flat, plot_variables):
    driver.cam.state.plot_profile(
        variable, rank=0, ax=axis, color='tab:orange', label='after 2 steps'
    )
print(driver.cam.state.summary(rank=0))

## 3. Add a variable and a Python physics process

Use an ordinary NumPy array for small rank-independent state: freeCAM copies the same shape and values into every rank's StatePool. Use `fc.Variable(dims=...)` when a field follows CAM's distributed grid and therefore has a rank-local shape. A `Physics` object can then be inserted directly into the live workflow.

In [ ]:
NLEV = driver.cam.state.T.metadata['shape'][1]
driver.cam.state.rh = np.zeros(NLEV)  # same 1-D profile on every rank

driver.cam.state.experiment_tracer = fc.Variable(
    dims=('pcols', 'pver', 'chunks'),
    units='kg kg-1',
    initial=0.0,
    aliases=('tracer',),
    standard_name='experiment_tracer',
)

class NotebookTracer(fc.Physics):
    name = 'notebook_tracer_source'
    after = 'dadadj'
    writes = ('tracer',)

    def tendency(self, fields, context):
        fields['tracer'][...] += 1.0e-6 * context.timestep_seconds

tracer_process = driver.cam.workflow.insert(NotebookTracer())
driver.cam.workflow

In [ ]:
# Run only this process; the model clock does not advance.
tracer_process.run()
print(driver.cam.state.experiment_tracer.stats(rank='global'))

# The same object controls its placement and on/off state.
tracer_process.move(before='deep_convection')
tracer_process.disable()
tracer_process.enable()

## 4. Load an original Fortran process at runtime

The same state syntax defines its inputs. freeCAM generates the `bind(C)` adapter, compiles the device `.so`, loads it on every rank, and inserts the process into the workflow.

In [ ]:
driver.cam.state.runtime_temperature = fc.Variable(
    dims=('nphys_local', 'pver'),
    units='K',
    initial=240.0,
    standard_name='runtime_plugin_temperature',
)
driver.cam.state.runtime_temperature_increment = fc.Variable(
    dims=(),
    units='K',
    initial=1.5,
    writable=False,
    standard_name='runtime_plugin_temperature_increment',
)

fortran_process = driver.cam.physics.install_fortran(
    driver.repo / 'examples/plugins/runtime_temperature_offset/device.yaml',
    project_root=driver.repo,
    process='runtime_temperature_offset',
    after='dadadj',
    unsafe=True,
)
fortran_process.run()
driver.cam.state.runtime_temperature.stats(rank='global')

## 5. Every supported physics interface

`driver.cam.physics` is a flat API: original module/subroutine nesting and CAM phases are implementation metadata, not part of the user path. The original UI had 36 workflow processes and 262 source-catalog-only interfaces. All 262 now have compiled StatePool pointer adapters. Together there are 276 unique source physics processes; 240 source devices load in this PI-CAM executable, including 226 of the former catalog-only set. The other 36 belong to inactive COSP, CARMA, or legacy-radiation configurations.

In [ ]:
physics = driver.cam.physics
display(physics)
print('coverage:', physics.coverage)
print('current-case callable:', physics.coverage['current_case_loadable'])
print('compiled adapters:', physics.coverage['compiled_process_adapters'])
print('former catalog-only interfaces:', physics.coverage['formerly_catalog_only_interfaces'])
print('former catalog-only adapters:', physics.coverage['catalog_adapters_compiled'])
print('former catalog-only loadable here:', physics.coverage['catalog_current_case_loadable'])

# Access by the original Fortran operation or by its readable workflow alias.
dadadj = physics.dadadj
print('original name:', dadadj.operation)
print('readable alias:', physics.dry_adjustment.operation)
dadadj_trace = dadadj.run()
print('dadadj:', dadadj_trace)

# Every source process uses the same flat namespace.
cloud_ice = physics.cloud_fraction_fice
evaporation = physics.zm_conv_evap
print('cloud_fraction_fice callable:', cloud_ice.runnable)
print('zm_conv_evap callable:', evaporation.runnable)

# Each process has its own control handle; restore the original state here.
physics.radiation.disable()
physics.radiation.enable()

## 6. Call an original process with Python objects

Calling a process binds inferred model inputs to StatePool, allocates output-only arguments, calls the generated adapter, and returns named output fields. The second example calls `calc_hltalt`, one of the newly generated source-catalog adapters. Isolated calls do not insert duplicate calls into the complete timestep.

In [ ]:
cloud_result = driver.cam.physics.cloud_fraction_fice()
saturation_result = driver.cam.physics.calc_hltalt(t=250.0)

# Named outputs are ordinary distributed StatePool fields.
print(cloud_result.fice.stats(rank='global'))
print(cloud_result.fsnow.stats(rank='global'))
print(saturation_result.hltalt.stats(rank='global'))

## 7. Remove runtime extensions and close the model

Processes must be removed before deleting fields they use. `driver.close()` finalizes CAM and releases the persistent PBS/MPI session.

In [ ]:
cloud_result.remove()
saturation_result.remove()
tracer_process.remove()
fortran_process.remove()

del driver.cam.state.rh
del driver.cam.state.experiment_tracer
del driver.cam.state.runtime_temperature
del driver.cam.state.runtime_temperature_increment

driver.close()
print('closed:', not driver.running)